# Seleção e Classificação de Variáveis

---

## Contexto

O projeto Sindusfarma tem como objetivo construir a espinha dorsal de dados da **Central de Pesquisas**, integrando fontes hoje dispersas (Excel, SurveyMonkey, Trello) em um banco centralizado na nuvem, com dashboards operacionais e estratégicos.

As bases disponibilizadas cobrem o ciclo de vida completo das pesquisas setoriais:

| Base | Registros | Descrição resumida |
|:---|---:|:---|
| `pesquisas.csv` | 366 | Ciclo de vida das pesquisas (solicitação → entrega) |
| `empresas.csv` | 779 | Cadastro das indústrias associadas |
| `clientes.csv` | 5.430 | Contatos/respondentes cadastrados |
| `respondentes.csv` | 18.735 | Registros de participação por pesquisa |
| `questoes.csv` | 231 | Perguntas dos questionários |
| `respostas.csv` | 20.607 | Respostas atômicas coletadas |

**Principal unidade de análise:** `pesquisa` — é em torno dela que orbitam empresas, respondentes, perguntas e respostas.  
Unidades secundárias relevantes: `empresa`, `cliente/respondente`.

---

## Objetivo desta tarefa

Identificar **pelo menos 10 variáveis relevantes** e classificá-las matematicamente segundo os tipos:

- **Identificadora** — chave primária ou candidata, usada para relacionar entidades
- **Categórica nominal** — categorias sem ordem natural
- **Categórica ordinal** — categorias com ordem significativa
- **Numérica discreta** — valores inteiros contáveis
- **Numérica contínua** — valores que podem assumir qualquer valor em um intervalo
- **Temporal** — datas, horas ou durações


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import MaxNLocator

# Configurações gerais de visualização
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Paleta de cores consistente com o projeto
CORES = {
    'identificadora':   '#1a3a5c',
    'cat_nominal':      '#2d6a9f',
    'cat_ordinal':      '#4a90c4',
    'num_discreta':     '#e07b39',
    'num_continua':     '#c0392b',
    'temporal':         '#27ae60',
}

print('Bibliotecas importadas com sucesso.')

In [ ]:
# ─── Caminhos das bases ────────────────────────────────────────────────────────
# Ajuste o caminho conforme a localização dos arquivos no seu ambiente
BASE_PATH = '../data/raw/'

pesquisas    = pd.read_csv(BASE_PATH + 'pesquisas.csv',    sep=';')
empresas     = pd.read_csv(BASE_PATH + 'empresas.csv',     sep=';')
clientes     = pd.read_csv(BASE_PATH + 'clientes.csv',     sep=';')
respondentes = pd.read_csv(BASE_PATH + 'respondentes.csv', sep=';')
questoes     = pd.read_csv(BASE_PATH + 'questoes.csv',     sep=';')
respostas    = pd.read_csv(BASE_PATH + 'respostas.csv',    sep=';')

print('Bases carregadas:')
for nome, df in [('pesquisas', pesquisas), ('empresas', empresas),
                 ('clientes', clientes), ('respondentes', respondentes),
                 ('questoes', questoes), ('respostas', respostas)]:
    print(f'  {nome:15s}: {df.shape[0]:6d} linhas × {df.shape[1]:2d} colunas')

---

## 1. Unidades de Análise

Antes de classificar variáveis, é preciso entender **quem** ou **o quê** cada tabela representa — essa é a **unidade de análise**, o nível em que os dados fazem sentido.


In [ ]:
unidades = [
    {
        'Unidade de Análise': 'Pesquisa',
        'Base principal': 'pesquisas.csv',
        'Chave': 'id',
        'Registros': len(pesquisas),
        'O que representa': 'Uma pesquisa setorial completa, do pedido à entrega — é o eixo central do projeto'
    },
    {
        'Unidade de Análise': 'Empresa',
        'Base principal': 'empresas.csv',
        'Chave': 'ID',
        'Registros': len(empresas),
        'O que representa': 'Uma indústria farmacêutica cadastrada; pode participar de várias pesquisas'
    },
    {
        'Unidade de Análise': 'Cliente (contato)',
        'Base principal': 'clientes.csv',
        'Chave': 'id_cliente',
        'Registros': len(clientes),
        'O que representa': 'Uma pessoa cadastrada, vinculada a uma empresa; pode responder múltiplas pesquisas'
    },
    {
        'Unidade de Análise': 'Participação',
        'Base principal': 'respondentes.csv',
        'Chave': '(id_pesq, id_empresa, id_cliente)',
        'Registros': len(respondentes),
        'O que representa': 'Um registro de participação de uma empresa/pessoa em uma pesquisa específica'
    },
    {
        'Unidade de Análise': 'Pergunta',
        'Base principal': 'questoes.csv',
        'Chave': 'id_pergunta',
        'Registros': len(questoes),
        'O que representa': 'Uma questão de um questionário, com tipo, alternativas e metadados'
    },
    {
        'Unidade de Análise': 'Elemento de resposta',
        'Base principal': 'respostas.csv',
        'Chave': 'id_resposta',
        'Registros': len(respostas),
        'O que representa': 'Uma linha atômica de resposta — uma alternativa marcada por um respondente para uma pergunta'
    },
]

df_unidades = pd.DataFrame(unidades)
print('=== Unidades de Análise identificadas ===')
print(df_unidades[['Unidade de Análise', 'Base principal', 'Chave', 'Registros']].to_string(index=False))

> **Nota sobre grão atômico de respostas.csv:** uma linha **não** representa um questionário respondido. Questões do tipo `caixas_de_selecao` e `matriz_avaliacao` geram múltiplas linhas para o mesmo respondente e pergunta. Por isso `COUNT(*)` nessa tabela mede *elementos de resposta*, não respondentes.

---

## 2. Classificação das Variáveis

A seguir, selecionamos **15 variáveis** de diferentes bases e as classificamos matematicamente. A escolha priorizou variáveis que alimentam os indicadores estratégicos do projeto (KPIs do Data Model Canvas).


In [ ]:
variaveis = [
    # ── Identificadoras ────────────────────────────────────────────────────
    {
        'Variável': 'pesquisas.id',
        'Base': 'pesquisas.csv',
        'Tipo Matemático': 'Identificadora',
        'Tipo Técnico (pandas)': 'int64',
        'Justificativa': 'Chave primária única da pesquisa; não tem sentido aritmético (não se soma IDs). É o elo central entre todas as bases.'
    },
    {
        'Variável': 'empresas.ID',
        'Base': 'empresas.csv',
        'Tipo Matemático': 'Identificadora',
        'Tipo Técnico (pandas)': 'int64',
        'Justificativa': 'Chave primária da empresa; referenciada em respondentes e respostas como chave estrangeira.'
    },
    # ── Categóricas Nominais ───────────────────────────────────────────────
    {
        'Variável': 'pesquisas.status',
        'Base': 'pesquisas.csv',
        'Tipo Matemático': 'Categórica nominal',
        'Tipo Técnico (pandas)': 'object',
        'Justificativa': 'Etapa atual da pesquisa ("Finalizada", "Em andamento", etc.). As categorias não têm ordem natural inequívoca — "Em tabulação" não é necessariamente maior ou menor que "Em análise".'
    },
    {
        'Variável': 'pesquisas.tipo',
        'Base': 'pesquisas.csv',
        'Tipo Matemático': 'Categórica nominal',
        'Tipo Técnico (pandas)': 'object',
        'Justificativa': 'Modalidade da pesquisa ("Pesquisa fixa anual", "Solicitação de associado", etc.). Sem ordenação natural entre os tipos.'
    },
    {
        'Variável': 'pesquisas.area',
        'Base': 'pesquisas.csv',
        'Tipo Matemático': 'Categórica nominal',
        'Tipo Técnico (pandas)': 'object',
        'Justificativa': 'Área temática da pesquisa ("RH", "Assuntos Regulatórios", etc.). Sem hierarquia implícita entre áreas.'
    },
    {
        'Variável': 'empresas.Nacionalidade',
        'Base': 'empresas.csv',
        'Tipo Matemático': 'Categórica nominal',
        'Tipo Técnico (pandas)': 'object',
        'Justificativa': 'Classificação da empresa como "Nacional" ou "Multinacional". Binária sem ordenação.'
    },
    {
        'Variável': 'clientes.cargo',
        'Base': 'clientes.csv',
        'Tipo Matemático': 'Categórica nominal',
        'Tipo Técnico (pandas)': 'object',
        'Justificativa': 'Cargo do contato ("Analista", "Gerente", "Diretor", etc.). Embora a hierarquia corporativa exista, a base não codifica essa ordem explicitamente — o campo é texto livre categorizado a posteriori.'
    },
    {
        'Variável': 'questoes.tipo',
        'Base': 'questoes.csv',
        'Tipo Matemático': 'Categórica nominal',
        'Tipo Técnico (pandas)': 'object',
        'Justificativa': 'Formato da pergunta ("multipla_escolha", "caixas_de_selecao", "matriz_avaliacao", etc.). Sem ordenação entre tipos de questão.'
    },
    # ── Categórica Ordinal ─────────────────────────────────────────────────
    {
        'Variável': 'pesquisas.complexidade',
        'Base': 'pesquisas.csv',
        'Tipo Matemático': 'Categórica ordinal',
        'Tipo Técnico (pandas)': 'object',
        'Justificativa': 'Nível de dificuldade da pesquisa: "baixa" < "média" < "alta". A ordem é semanticamente clara e pode ser codificada numericamente (1, 2, 3) para análises.'
    },
    # ── Numéricas Discretas ────────────────────────────────────────────────
    {
        'Variável': 'pesquisas.participantes',
        'Base': 'pesquisas.csv',
        'Tipo Matemático': 'Numérica discreta',
        'Tipo Técnico (pandas)': 'float64',
        'Justificativa': 'Número de empresas participantes de uma pesquisa. Inteiro não negativo; admite soma, média e outras operações aritméticas.'
    },
    {
        'Variável': 'questoes.qtd_alternativas',
        'Base': 'questoes.csv',
        'Tipo Matemático': 'Numérica discreta',
        'Tipo Técnico (pandas)': 'float64',
        'Justificativa': 'Quantidade de opções de uma pergunta. Sempre inteiro, não pode ser fracionário.'
    },
    # ── Numéricas Contínuas ────────────────────────────────────────────────
    {
        'Variável': 'pesquisas.dias_uteis_para_entrega',
        'Base': 'pesquisas.csv',
        'Tipo Matemático': 'Numérica contínua',
        'Tipo Técnico (pandas)': 'float64',
        'Justificativa': 'Dias úteis totais do ciclo da pesquisa. Embora registrado como inteiro, o conceito é uma duração contínua; admite média, desvio padrão e percentis como métricas de desempenho.'
    },
    {
        'Variável': 'empresas.% de mercado',
        'Base': 'empresas.csv',
        'Tipo Matemático': 'Numérica contínua',
        'Tipo Técnico (pandas)': 'object (requer conversão)',
        'Justificativa': 'Participação percentual de mercado da empresa. Valor real em [0, 100]; armazenado como texto ("8,25%") — requer limpeza antes de operações numéricas.'
    },
    # ── Temporais ──────────────────────────────────────────────────────────
    {
        'Variável': 'pesquisas.data_solicitacao',
        'Base': 'pesquisas.csv',
        'Tipo Matemático': 'Temporal',
        'Tipo Técnico (pandas)': 'object (requer parse)',
        'Justificativa': 'Data de início do ciclo da pesquisa. Permite calcular duração, distribuição temporal e tendências.'
    },
    {
        'Variável': 'respondentes.hora_inicio',
        'Base': 'respondentes.csv',
        'Tipo Matemático': 'Temporal',
        'Tipo Técnico (pandas)': 'object (requer parse)',
        'Justificativa': 'Data e hora de início do preenchimento do questionário. Permite calcular tempo de resposta e padrões de acesso.'
    },
]

df_vars = pd.DataFrame(variaveis)
print(df_vars[['Variável', 'Base', 'Tipo Matemático', 'Tipo Técnico (pandas)']].to_string(index=False))

---

## 3. Visualização: distribuição das variáveis por tipo


In [ ]:
contagem_tipos = df_vars['Tipo Matemático'].value_counts()

MAPA_CORES = {
    'Identificadora':       CORES['identificadora'],
    'Categórica nominal':   CORES['cat_nominal'],
    'Categórica ordinal':   CORES['cat_ordinal'],
    'Numérica discreta':    CORES['num_discreta'],
    'Numérica contínua':    CORES['num_continua'],
    'Temporal':             CORES['temporal'],
}

cores_barras = [MAPA_CORES[t] for t in contagem_tipos.index]

fig, ax = plt.subplots(figsize=(9, 4))
barras = ax.barh(contagem_tipos.index, contagem_tipos.values, color=cores_barras, height=0.6)

for barra, valor in zip(barras, contagem_tipos.values):
    ax.text(valor + 0.05, barra.get_y() + barra.get_height() / 2,
            str(valor), va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Quantidade de variáveis')
ax.set_title('Distribuição das 15 variáveis selecionadas por tipo matemático', fontsize=12, fontweight='bold')
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.set_xlim(0, contagem_tipos.max() + 1.2)
plt.tight_layout()
plt.savefig('fig_01_tipos_variaveis.png', bbox_inches='tight')
plt.show()
print('Figura salva: fig_01_tipos_variaveis.png')

---

## 4. Verificação rápida dos tipos nativos (pandas vs. tipo semântico)

Uma inconsistência comum nos dados recebidos é o **tipo técnico divergir do tipo semântico**. O exemplo mais crítico aqui são as colunas de percentual de mercado armazenadas como `object` (texto).


In [ ]:
# Verificação de inconsistências de tipo técnico vs semântico
inconsistencias = [
    ('pesquisas.csv', 'part_mercado', 'object', 'Numérica contínua', '"58%" — texto com vírgula decimal e símbolo %'),
    ('empresas.csv', '% de mercado', 'object', 'Numérica contínua', '"8,25%" — texto com vírgula decimal e símbolo %'),
    ('respondentes.csv', 'market_share', 'object', 'Numérica contínua', '"0,38%" — texto com vírgula decimal e símbolo %'),
    ('pesquisas.csv', 'data_solicitacao', 'object', 'Temporal', 'Datas em texto no formato dd/mm/aaaa'),
    ('respondentes.csv', 'hora_inicio', 'object', 'Temporal', 'Timestamps em texto; 95,7% dos registros são nulos'),
    ('clientes.csv', 'id_cliente', 'object', 'Identificadora', 'Contém "s/i" (sentinela) além de valores numéricos — tipo misto'),
]

df_inconsistencias = pd.DataFrame(inconsistencias,
    columns=['Base', 'Campo', 'Tipo técnico atual', 'Tipo semântico esperado', 'Problema'])

print('=== Inconsistências tipo técnico × tipo semântico ===')
print(df_inconsistencias.to_string(index=False))

print()
print('Impacto: qualquer operação aritmética direta nessas colunas retornará erro ou resultado incorreto.')
print('Recomendação: converter na camada tipada do pipeline, preservando o valor original para auditoria.')

---

## 5. Tabela-resumo final


In [ ]:
print('=== Tabela-resumo: 15 variáveis classificadas ===')
print()
for _, row in df_vars.iterrows():
    print(f"[{row['Tipo Matemático']:22s}] {row['Variável']:45s} | {row['Base']}")

## Conclusões

1. **6 tipos matemáticos** foram identificados nas bases do Sindusfarma, com predominância de variáveis **categóricas nominais** — o que é esperado em um sistema de gestão de pesquisas com muitos atributos descritivos.

2. **Atenção especial** é necessária para:
   - As três colunas de **percentual de mercado** (`part_mercado`, `% de mercado`, `market_share`), que são numéricas contínuas armazenadas como texto. Qualquer cálculo de representatividade depende da conversão dessas colunas.
   - As **colunas temporais** armazenadas como texto, que requerem parsing consistente antes de calcular durações ou distribuições.
   - A variável `complexidade` é a **única ordinal** identificada — ela deve ser codificada como `baixa=1, média=2, alta=3` em análises que exijam ordenação.

3. As variáveis numéricas contínuas (`dias_uteis_para_entrega`, `% de mercado`, `participantes`) são as que alimentam diretamente os KPIs de **tempo de ciclo** e **representatividade** definidos no Data Model Canvas.
